In [1]:
import pandas as pd
import numpy as np

In [2]:
train_labels = pd.read_csv('Data/train.csv')
test_labels = pd.read_csv('Data/dev.csv')

In [3]:
y_train = np.array([label for label in train_labels['PHQ8_Binary']])
y_test = np.array([label for label in test_labels['PHQ8_Binary']])

# Text Model

In [4]:
train_text = []
for label in train_labels['Participant_ID']:
    sample = pd.read_csv(f"Data/train/{label}_P/{label}_TRANSCRIPT.csv", sep = "\t")
    train_text.append(sample)

In [5]:
test_text = []
for label in test_labels['Participant_ID']:
    sample = pd.read_csv(f"Data/test/{label}_P/{label}_TRANSCRIPT.csv", sep = "\t")
    test_text.append(sample)

In [6]:
for sample in train_text:
    sample["total_time"] = sample["stop_time"] - sample["start_time"]
for sample in test_text:
    sample["total_time"] = sample["stop_time"] - sample["start_time"]

In [7]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
encoder.fit(train_text[0]["speaker"])
for sample in test_text:
    sample["speaker"] = encoder.transform(sample["speaker"])
for sample in train_text:
    sample["speaker"] = encoder.transform(sample["speaker"])

In [8]:
from sklearn.preprocessing import StandardScaler
Numerics = []
for sample in test_text:
    Numerics.append(sample.loc[:,["start_time", "stop_time", "total_time"]])
Numerics = pd.concat(Numerics)
scale = StandardScaler()
scale.fit(Numerics)
for sample in test_text:
    sample.loc[:,["start_time", "stop_time", "total_time"]] = scale.transform(sample.loc[:,["start_time", "stop_time", "total_time"]])
for sample in train_text:
    sample.loc[:,["start_time", "stop_time", "total_time"]] = scale.transform(sample.loc[:,["start_time", "stop_time", "total_time"]])

In [9]:
from transformers import BertTokenizer, BertModel
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.eval()
model = model.to(device)
def convert(text):
    text = str(text)
    tokens = tokenizer(
    text,
    return_tensors='pt',
    padding=True,
    truncation=True,
    max_length=256
    )
    tokens = {k: v.to(device) for k, v in tokens.items()}
    with torch.no_grad():
        res = model(**tokens)
    return res.last_hidden_state[:, 0, :].detach().cpu().numpy()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
for i,sample in enumerate(train_text):
    if sum(sample.isna().sum()) != 0 :
        sample.dropna(inplace=True)
for i,sample in enumerate(test_text):
    if sum(sample.isna().sum()) != 0 :
        sample.dropna(inplace=True)

In [11]:
for sample in train_text:
    sample['value'] = sample['value'].apply(convert)
for sample in test_text:
    sample['value'] = sample['value'].apply(convert)

In [12]:
for i,sample in enumerate(train_text):
    single = np.array([
        np.concatenate([
            np.array([t, s], dtype=np.float32),
            np.asarray(v, dtype=np.float32).ravel()
        ])
        for t, s, v in zip(sample['total_time'], sample['speaker'], sample['value'])
    ], dtype=np.float32)
    train_text[i] = single
for i,sample in enumerate(test_text):
    single = np.array([
        np.concatenate([
            np.array([t, s], dtype=np.float32),
            np.asarray(v, dtype=np.float32).ravel()
        ])
        for t, s, v in zip(sample['total_time'], sample['speaker'], sample['value'])
    ], dtype=np.float32)
    test_text[i] = single

In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
train_text = pad_sequences(
    train_text,
    maxlen=473,
    padding='post',      # zeros added at END of each sample
    truncating='post',   # if truncating needed, cut from END
    dtype='float32'
)
test_text = pad_sequences(
    test_text,
    maxlen=473,
    padding='post',      # zeros added at END of each sample
    truncating='post',   # if truncating needed, cut from END
    dtype='float32'
)

In [14]:
train_text.shape

(107, 473, 770)

# Audio

In [15]:
train_audio = []
for rank in train_labels['Participant_ID']:
    sample = pd.read_csv(f'Data/train/{rank}_P/{rank}_COVAREP.csv', header=None)
    sample['index'] = sample.index
    time = pd.read_csv(f'Data/train/{rank}_P/{rank}_TRANSCRIPT.csv', sep='\t')
    time = time[time['speaker'] == 'Ellie']
    for start,stop in zip(time['start_time'], time['stop_time']):
        sample.drop(sample[sample['index'].isin(range(int(100*start),int(100*stop)))].index, inplace = True)
    sample = sample[sample[1] == 1]
    train_audio.append(sample.drop(columns = ['index']))

In [16]:
test_audio = []
for rank in test_labels['Participant_ID']:
    sample = pd.read_csv(f'Data/test/{rank}_P/{rank}_COVAREP.csv', header=None)
    sample['index'] = sample.index
    time = pd.read_csv(f'Data/test/{rank}_P/{rank}_TRANSCRIPT.csv', sep='\t')
    time = time[time['speaker'] == 'Ellie']
    for start,stop in zip(time['start_time'], time['stop_time']):
        sample.drop(sample[sample['index'].isin(range(int(100*start),int(100*stop)))].index, inplace = True)
    sample = sample[sample[1] == 1]
    test_audio.append(sample.drop(columns = ['index']))

In [17]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
scale = pd.concat([sample.iloc[:,[0]] for sample in train_audio])
sc.fit(scale)

,copy,True
,with_mean,True
,with_std,True


In [18]:
for sample in train_audio:
    sample.iloc[:,[0]] = sc.transform(sample.iloc[:,[0]])
    sample.drop(columns=sample.columns[1], inplace = True)
for sample in test_audio:
    sample.iloc[:,[0]] = sc.transform(sample.iloc[:,[0]])
    sample.drop(columns=sample.columns[1], inplace = True)

In [19]:
for i,sample in enumerate(train_audio):
    train_audio[i] = sample.groupby(sample.index//25).mean()
for i,sample in enumerate(test_audio):
    test_audio[i] = sample.groupby(sample.index//25).mean()

In [20]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
train_audio = pad_sequences(train_audio, maxlen=3400, dtype='float32', padding='post', value=0.0)
test_audio = pad_sequences(test_audio, maxlen=3400, dtype='float32', padding='post', value=0.0)

# Video

In [21]:
train_video = []
for add in train_labels['Participant_ID']:
    au = pd.read_csv(f'Data/train/{add}_P/{add}_CLNF_AUs.txt')
    gaze = pd.read_csv(f'Data/train/{add}_P/{add}_CLNF_gaze.txt')
    pose = pd.read_csv(f'Data/train/{add}_P/{add}_CLNF_pose.txt')
    sample = au.merge(gaze).merge(pose)
    sample = sample[sample[' confidence'] > 0.75]
    sample = sample[sample[' success'] == 1]
    sample.drop(columns=[ 'frame', ' timestamp', ' confidence', ' success'], inplace=True)
    train_video.append(sample)

In [22]:
test_video = []
for add in test_labels['Participant_ID']:
    au = pd.read_csv(f'Data/test/{add}_P/{add}_CLNF_AUs.txt')
    gaze = pd.read_csv(f'Data/test/{add}_P/{add}_CLNF_gaze.txt')
    pose = pd.read_csv(f'Data/test/{add}_P/{add}_CLNF_pose.txt')
    sample = au.merge(gaze).merge(pose)
    sample = sample[sample[' confidence'] > 0.75]
    sample = sample[sample[' success'] == 1]
    sample.drop(columns=[ 'frame', ' timestamp', ' confidence', ' success'], inplace=True)
    test_video.append(sample)

In [23]:
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
sc.fit(pd.concat(train_video, axis=0, ignore_index=True))
for i in range(len(train_video)):
    train_video[i] = sc.transform(train_video[i])
for i in range(len(test_video)):
    test_video[i] = sc.transform(test_video[i])

In [24]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

train_video = pad_sequences(train_video, maxlen=52256, padding='post', value=0.0, dtype='float32')
test_video = pad_sequences(test_video, maxlen=52256, padding='post', value=0.0, dtype='float32')

# Models

In [25]:
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
import tensorflow as tf

text_input = tf.keras.Input(shape=train_text.shape[1:], name = 'text')
text = Masking(mask_value=0.0)(text_input)
text = Bidirectional(LSTM(64, return_sequences=True))(text)
text = Dropout(0.3)(text)
text = Bidirectional(LSTM(32))(text)

In [26]:
audio_input = tf.keras.Input(shape = train_audio.shape[1:],name = 'audio')
audio = Masking(mask_value=0.0)(audio_input)
audio = LSTM(64, return_sequences=True)(audio)
audio = Dropout(0.3)(audio)
audio = Bidirectional(LSTM(32))(audio)

In [27]:
video_input = tf.keras.Input(shape = train_video.shape[1:],name = 'video')
video = Masking(mask_value=0.0)(video_input)
video = LSTM(64, return_sequences=True)(video)
video = Dropout(0.3)(video)
video = Bidirectional(LSTM(32))(video)

In [28]:
combine = Concatenate()([text,audio,video])

x = Dense(64, activation='relu')(combine)
x = Dropout(0.3)(x)
x = Dense(16, activation='relu')(x)
out = Dense(1, activation='sigmoid')(x)

In [29]:
model = Model(
    inputs = [text_input, audio_input, video_input],
    outputs = out
)

In [30]:
model.compile(
    optimizer = 'adam',
    loss = 'binary_crossentropy',
    metrics= ['accuracy']
)

In [31]:
model.fit(
    [train_text, train_audio, train_video],
    y_train,
    epochs=10,
    batch_size=4,
    validation_split=0.2
)

Epoch 1/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 7064s 323s/step - accuracy: 0.6824 - loss: 0.6886 - val_accuracy: 0.9091 - val_loss: 0.5287
Epoch 2/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 5469s 249s/step - accuracy: 0.6588 - loss: 0.6504 - val_accuracy: 0.9091 - val_loss: 0.4958
Epoch 3/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 5625s 256s/step - accuracy: 0.7294 - loss: 0.5919 - val_accuracy: 0.8182 - val_loss: 0.5341
Epoch 4/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 5641s 256s/step - accuracy: 0.7176 - loss: 0.5092 - val_accuracy: 0.7727 - val_loss: 0.4841
Epoch 5/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 5730s 260s/step - accuracy: 0.9059 - loss: 0.3386 - val_accuracy: 0.7727 - val_loss: 0.4192
Epoch 6/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 5625s 255s/step - accuracy: 1.0000 - loss: 0.1099 - val_accuracy: 0.7273 - val_loss: 0.5194
Epoch 7/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 5597s 255s/step - accuracy: 1.0000 - loss: 0.0355 - val_accuracy: 0.6364 - val_loss: 0.6702
Epoch 8/10
22/22 ━━━━━━━━━━━━━━━━━━━━ 5608s 255s/step - accuracy: 1.0000 - loss: 0.0132 - 

In [32]:
pred = model.predict([test_text, test_audio, test_video])

2/2 ━━━━━━━━━━━━━━━━━━━━ 42s 10s/step


In [33]:
pred

array([[2.66053714e-02],
       [1.24813095e-02],
       [9.43527615e-04],
       [9.94555593e-01],
       [6.64954543e-01],
       [5.60282788e-04],
       [3.88679579e-02],
       [1.94103352e-03],
       [4.83486941e-03],
       [6.85175419e-01],
       [2.60157511e-03],
       [9.98055398e-01],
       [9.96907473e-01],
       [4.10517096e-04],
       [1.23162009e-02],
       [3.26437381e-04],
       [9.87330556e-01],
       [1.09943645e-02],
       [7.13853717e-01],
       [6.56738004e-04],
       [9.43176091e-01],
       [4.85349819e-03],
       [9.98022497e-01],
       [9.94744122e-01],
       [4.58436832e-03],
       [7.52580941e-01],
       [9.71126139e-01],
       [1.13887792e-04],
       [6.72128499e-02],
       [6.23680234e-01],
       [7.54532084e-05],
       [2.56269693e-01],
       [4.71948006e-05],
       [3.67006578e-04],
       [8.65104143e-03]], dtype=float32)

In [63]:
y_pred = [1 if p > 0.5 else 0 for p in pred]

In [64]:
from sklearn.metrics import accuracy_score, confusion_matrix
print(confusion_matrix(y_test,y_pred))
print((accuracy_score(y_test,y_pred)))

[[17  6]
 [ 5  7]]
0.6857142857142857
